# Inter-Annotator Agreement — Abstract Integrity (first 1,000)

Two human annotators (A1, A2) and two LLMs (Claude Opus 4.6, Codex GPT-5.4) independently labelled abstracts.\
Question: *"Is this a complete and meaningful scientific abstract?"* (Yes / No)\
This notebook compares their answers on the first 1,000 entries.

## Human annotations

In [1]:
import json
import numpy as np
from sklearn.metrics import cohen_kappa_score, confusion_matrix

N = 1000  # first N entries to compare

with open("output/integrity_A1.json") as f:
    a1_data = json.load(f)
with open("output/integrity_A2.json") as f:
    a2_data = json.load(f)
with open("output/integrity_tasks.json") as f:
    tasks_data = json.load(f)

entries = {e["entry_id"]: e for e in tasks_data["entries"][:N]}

a1  = np.array([a1_data["answers"][str(i)]["answer"] for i in range(N)])
a2 = np.array([a2_data["answers"][str(i)]["answer"] for i in range(N)])

print(f"Entries compared: {N}")
print(f"A1  — Yes: {a1.sum()}, No: {N - a1.sum()}  (bad-abstract rate: {(N - a1.sum()) / N:.1%})")
print(f"A2 — Yes: {a2.sum()}, No: {N - a2.sum()}  (bad-abstract rate: {(N - a2.sum()) / N:.1%})")

Entries compared: 1000
A1  — Yes: 932, No: 68  (bad-abstract rate: 6.8%)
A2 — Yes: 769, No: 231  (bad-abstract rate: 23.1%)


## Agreement statistics

In [2]:
agree = (a1 == a2).sum()
raw_agreement = agree / N
kappa = cohen_kappa_score(a1, a2)

# Confusion matrix: rows = A1, cols = A2
#   [[both_No,  V_No  S_Yes],
#    [V_Yes S_No, both_Yes ]]
cm = confusion_matrix(a1, a2, labels=[False, True])

print(f"Raw agreement : {raw_agreement:.1%}  ({agree}/{N})")
print(f"Cohen's kappa : {kappa:.3f}")
print()
print("Contingency table (rows=A1, cols=A2):")
print(f"{'':>20s} {'A2=No':>10s} {'A2=Yes':>10s}")
print(f"{'A1=No':>20s} {cm[0,0]:>10d} {cm[0,1]:>10d}")
print(f"{'A1=Yes':>20s} {cm[1,0]:>10d} {cm[1,1]:>10d}")
print()
n_disagree = (a1 != a2).sum()
print(f"Disagreements: {n_disagree}  (A1=Yes/A2=No: {cm[1,0]}, A1=No/A2=Yes: {cm[0,1]})")

Raw agreement : 83.5%  (835/1000)
Cohen's kappa : 0.383

Contingency table (rows=A1, cols=A2):
                      A2=No A2=Yes
            A1=No         67          1
           A1=Yes        164        768

Disagreements: 165  (A1=Yes/A2=No: 164, A1=No/A2=Yes: 1)


## Disagreements

### A1 = Yes (good), A2 = No (bad)

In [3]:
mask_vy_sn = a1 & ~a2  # A1=Yes, A2=No
ids_vy_sn = np.where(mask_vy_sn)[0]

for idx in ids_vy_sn:
    e = entries[idx]
    abstract = e["abstract"]
    if len(abstract) > 500:
        abstract = abstract[:500] + " [...]"
    print(f"--- Entry {idx}  |  {e['paper_id']}  |  len={e['abstract_length']}")
    print(f"    Title: {e['title']}")
    print(f"    Abstract: {abstract}")
    print(f"    A1=Yes  A2=No")
    print()

--- Entry 1  |  https://openalex.org/W2013866881  |  len=407
    Title: Summated Cortical Evoked Response Testing in the Deafferented Primate
    Abstract: Dorsal rhizotomy from C 2 or C 3 to T 4 in the primate results in failure to elicit summated cortical responses from systematic stimulation of the appropriate peripheral nerves. Under these conditions there is thus no evidence of sensory input into the cerebral cortex. A nonclassical mechanism must therefore be operational to explain the extensive purposive movements observed in the deafferented animals.
    A1=Yes  A2=No

--- Entry 3  |  https://openalex.org/W2915934899  |  len=272
    Title: Probing the absence of third phase formation during the extraction of trivalent metal ions in an ionic liquid medium
    Abstract: In contrast to molecular diluents, diglycolamide (T2EHDGA) and carbamoylmethyl-phosphine oxide (CMPO) extractants diluted in an ionic liquid diluent minimize aggregation upon nitric acid extraction and prevent thir

### A1 = No (bad), A2 = Yes (good)

In [4]:
mask_vn_sy = ~a1 & a2  # A1=No, A2=Yes
ids_vn_sy = np.where(mask_vn_sy)[0]

for idx in ids_vn_sy:
    e = entries[idx]
    abstract = e["abstract"]
    if len(abstract) > 500:
        abstract = abstract[:500] + " [...]"
    print(f"--- Entry {idx}  |  {e['paper_id']}  |  len={e['abstract_length']}")
    print(f"    Title: {e['title']}")
    print(f"    Abstract: {abstract}")
    print(f"    A1=No  A2=Yes")
    print()

--- Entry 410  |  https://openalex.org/W2810843202  |  len=888
    Title: LEVERAGE, PROFITABILITAS, UKURAN PERUSAHAAN, PENGUNGKAPAN CORPORATE SOCIAL RESPONSIBILITY DENGAN PENDEKATAN KAUSALITAS
    Abstract: Management of natural resources and environment who is not responsible the main issues disclosure of corporate social responsibility (Wahba &amp; Elsayed, 2015). This study aims to examine the effect of leverage, profitability, and size toward the disclosure of corporate social responsibility. The samples of this study are 51 companies listed in Indonesia Stock Exchange selected by using purposive sampling method. Data analysis method used is panel regression model. The result this study tested  [...]
    A1=No  A2=Yes



---

# LLM Annotations — Claude Opus 4.6 and Codex GPT-5.4

Both models annotated the same first 1,000 entries with the same question:
*"Is this a complete and meaningful scientific abstract?"*

- **Claude Opus 4.6** (1M context): in-context subagent, 10 batches of 100, no additional criteria
- **Codex GPT-5.4**: rubric-based annotation with scripted candidate filtering, reasoning effort = high

In [5]:
with open("output/integrity_claude.json") as f:
    claude_data = json.load(f)
with open("output/integrity_Codex_GPT54_first1000.json") as f:
    codex_data = json.load(f)

claude = np.array([claude_data["answers"][str(i)]["answer"] for i in range(N)])
codex  = np.array([codex_data["answers"][str(i)]["answer"]  for i in range(N)])

print(f"{'Annotator':<16s} {'Yes':>5s} {'No':>5s} {'Rejection rate':>15s}")
print("-" * 45)
for name, arr in [("A1", a1), ("A2", a2), ("Claude", claude), ("Codex", codex)]:
    n_no = N - arr.sum()
    print(f"{name:<16s} {arr.sum():>5d} {n_no:>5d} {n_no/N:>14.1%}")

Annotator          Yes    No  Rejection rate
---------------------------------------------
A1              932    68           6.8%
A2             769   231          23.1%
Claude             876   124          12.4%
Codex              939    61           6.1%


## Pairwise agreement (all four annotators)

In [6]:
from itertools import combinations

annotators = {"A1": a1, "A2": a2, "Claude": claude, "Codex": codex}

print(f"{'Pair':<22s} {'Agreement':>10s} {'Kappa':>8s}  {'Both Yes':>9s} {'Both No':>8s} {'A=Y B=N':>8s} {'A=N B=Y':>8s}")
print("-" * 80)
for (n1, a1), (n2, a2) in combinations(annotators.items(), 2):
    agree = (a1 == a2).sum()
    kappa = cohen_kappa_score(a1, a2)
    cm = confusion_matrix(a1, a2, labels=[False, True])
    both_no, a_no_b_yes = cm[0, 0], cm[0, 1]
    a_yes_b_no, both_yes = cm[1, 0], cm[1, 1]
    print(f"{n1+' vs '+n2:<22s} {agree/N:>9.1%} {kappa:>8.3f}  {both_yes:>9d} {both_no:>8d} {a_yes_b_no:>8d} {a_no_b_yes:>8d}")

Pair                    Agreement    Kappa   Both Yes  Both No  A=Y B=N  A=N B=Y
--------------------------------------------------------------------------------
A1 vs A2            83.5%    0.383        768       67      164        1
A1 vs Claude            93.0%    0.600        869       61       63        7
A1 vs Codex             97.7%    0.809        924       53        8       15
A2 vs Claude           86.5%    0.547        755      110       14      121
A2 vs Codex            82.6%    0.340        767       59        2      172
Claude vs Codex            93.1%    0.594        873       58        3       66


## LLM-rejected entries that both humans accepted

Cases where an LLM said "No" but both A1 and A2 said "Yes". These are potential false positives by the LLM — or subtle quality issues the humans missed.

In [7]:
both_human_yes = a1 & a2

claude_fp = ~claude & both_human_yes
codex_fp  = ~codex  & both_human_yes
either_fp = (claude_fp | codex_fp)

print(f"Both humans = Yes:                           {both_human_yes.sum()}")
print(f"Claude rejects (both humans = Yes):          {claude_fp.sum()}")
print(f"Codex rejects (both humans = Yes):           {codex_fp.sum()}")
print(f"Either LLM rejects (both humans = Yes):      {either_fp.sum()}")
print(f"Both LLMs reject (both humans = Yes):        {(claude_fp & codex_fp).sum()}")
print()

# Show the entries
for label, mask in [("Claude only", claude_fp & ~codex_fp),
                    ("Codex only", codex_fp & ~claude_fp),
                    ("Both LLMs", claude_fp & codex_fp)]:
    ids = np.where(mask)[0]
    if len(ids) == 0:
        continue
    print(f"\n{'='*80}")
    print(f"  {label} rejects — both humans said Yes  ({len(ids)} entries)")
    print(f"{'='*80}")
    for idx in ids:
        e = entries[idx]
        abstract = e["abstract"]
        if len(abstract) > 400:
            abstract = abstract[:400] + " [...]"
        print(f"\n--- Entry {idx}  |  len={e['abstract_length']}")
        print(f"    Title: {e['title']}")
        print(f"    Abstract: {abstract}")
        print(f"    A1=Yes  A2=Yes  Claude={'Yes' if claude[idx] else 'No'}  Codex={'Yes' if codex[idx] else 'No'}")

Both humans = Yes:                           768
Claude rejects (both humans = Yes):          14
Codex rejects (both humans = Yes):           2
Either LLM rejects (both humans = Yes):      16
Both LLMs reject (both humans = Yes):        0


  Claude only rejects — both humans said Yes  (14 entries)

--- Entry 137  |  len=1003
    Title: Word Order and Incremental Update
    Abstract: The central claim of this paper is that surface-faithful word-by-word update is feasible and desirable, even in languages where word order is supposedly free. As a first step, in sections 1 and 2, I review an argument from Bittner 2001a that semantic composition is not a static process, as in PTQ, but rather a species of anaphoric bridging. But in that case the context-setting role of word order s [...]
    A1=Yes  A2=Yes  Claude=No  Codex=Yes

--- Entry 340  |  len=309
    Title: A new support vector machine optimized by improved particle swarm optimization and its application
    Abstract: 改进的粒子群优化(PSO )

## Four-way agreement breakdown

In [8]:
from collections import Counter

# Build label tuples: (A1, A2, Claude, Codex) as Y/N strings
patterns = Counter()
for i in range(N):
    pat = tuple("Y" if a[i] else "N" for a in [a1, a2, claude, codex])
    patterns[pat] += 1

print(f"{'A1':>6s} {'A2':>7s} {'Claude':>7s} {'Codex':>6s} {'Count':>7s}")
print("-" * 40)
for pat, count in patterns.most_common():
    print(f"{pat[0]:>6s} {pat[1]:>7s} {pat[2]:>7s} {pat[3]:>6s} {count:>7d}")

# Unanimous agreement
all_yes = (a1 & a2 & claude & codex).sum()
all_no  = (~a1 & ~a2 & ~claude & ~codex).sum()
print(f"\nUnanimous Yes: {all_yes}  |  Unanimous No: {all_no}  |  Total unanimous: {all_yes + all_no} ({(all_yes + all_no)/N:.1%})")

 A1  A2  Claude  Codex   Count
----------------------------------------
     Y       Y       Y      Y     752
     Y       N       Y      Y     115
     N       N       N      N      52
     Y       N       N      Y      43
     Y       Y       N      Y      14
     N       N       N      Y       9
     Y       N       N      N       6
     N       N       Y      Y       5
     Y       Y       Y      N       2
     N       Y       Y      Y       1
     N       N       Y      N       1

Unanimous Yes: 752  |  Unanimous No: 52  |  Total unanimous: 804 (80.4%)
